# NetCDF File Reformat for ArcGIS Pro Tutorial Notebook

## Summary
The purpose of this notebooks is to show users of suborbital campaign data in NetCDF format how to reformat input files to make them useable in ArcGIS Pro as a Multidimensional Dataset. Previously, users have given feedback that NetCDF files do not plot properly as Multidimensional Datasets in ArcGIS Pro - this notebook highlights the root cause and demonstrates how to restructure the file to work seemlessly in ArcGIS Pro. 

## Prerequisites
**Please create a "nc_files" folder to house input NetCDF files, and create a "new_nc_files" folder to house converted NetCDF files.**
- netCDF4
- numpy
- pathlib
- datetime
- tqdm


## Notebook Author/Affiliation
Gabriel Mojica/Atmospheric Science Data Center (ASDC)

## Steps


<font size="5">Import required packages</font><p></p>
<font size="4">For packages you don't have, run a <i>pip install</i> to add them to your machine</font>

In [1]:
from netCDF4 import Dataset
import numpy as np
from pathlib import Path
from datetime import datetime, time
import tqdm

<font size="5">Define directories to iterate through and write to. Ensure you have the source and output directory folders created</font>

In [2]:
source_dir = 'nc_files/'
output_dir = 'new_nc_files/'
input_path = Path(source_dir)
input_list = list(input_path.iterdir())
output_path = Path(output_dir)
output_list = list(output_path.iterdir())

<font size="5">Build function to copy dimensions, attributes, and variables from <i>src_group</i> to <i>dst_group</i></font><p></p>
<font size="4">Copy group attributes with the setncatts() function</font><p></p>
<font size="4">Copy dimensions with the createDimension() function</font><p></p>
<font size="4">Copy variables with condition to change "flag" variables from <i>float64</i> to <i>int32</i> datatypes</font><p></p>
- <font size="3">Check if variable name contains "flag" and is <i>float64</i>, change to <i>int32</i> then fix NaN values</font>
- <font size="3">Create the variables in the newly built group</font>
- <font size="3">Copy variable attributes to the new variables</font>
- <font size="3">Write in the data</font>
<p><font size="4">Create the subgroups with the createGroup() function</font></p>


In [3]:
def copy_nc_structure(src_group, dst_group):
    
    dst_group.setncatts(src_group.__dict__)

    for name, dimension in src_group.dimensions.items():
        dst_group.createDimension(
            name, (len(dimension) if not dimension.isunlimited() else None)
        )

    for name, var in src_group.variables.items():
        if 'flag' in name.lower() and var.datatype == np.float64:
            datatype = np.int32
            data = np.nan_to_num(var[:], nan=-1).astype(np.int32)
        else:
            datatype = var.datatype
            data = var[:]

        new_var = dst_group.createVariable(name, datatype, var.dimensions, zlib=True)

        new_var.setncatts(var.__dict__)

        new_var[:] = data

    for group_name, sub_group in src_group.groups.items():
        new_sub_group = dst_group.createGroup(group_name)
        copy_nc_structure(sub_group, new_sub_group)



<font size="5">Once you create the function, you can now run it using an <i>if</i> loop</font>

In [ ]:
if len(input_list) >0:
    print(f"Processing {len(input_list)} NetCDF Files.")
    for file in tqdm.tqdm(input_list):
        with Dataset(str(input_path) + '/' + file.name, 'r') as src, Dataset(str(output_path) + '/' + file.name, 'w') as dst:
            copy_nc_structure(src, dst)
    print(f"Your cleaned NetCDF files have been created in {output_path}.")
else:
    print(f"There are no NetCDF files in {input_list}.")

<font size="5">Now that you've created the new NetCDF files, you can delete the files in your <i>nc_files</i> directory by running the following cell:</font>

In [ ]:
for item in tqdm.tqdm(input_list):
    if item.is_file():
        item.unlink()
print("Your old NetCDF files have been deleted.")